In [1]:
import pandas as pd
from tqdm import tqdm
import json,re
import os
import matplotlib.pyplot as plt
import seaborn as sns
import shutil
from PIL import Image
import math

def read_jsonl(file_path):
    data = []
    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            try:
                json_object = json.loads(line.strip())
                data.append(json_object)
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON: {e}")
                continue
                # 如果选择抛出异常，使用下面这行
                # raise
    return data

In [2]:
result_path = "/home/aiqihang.aqh/Appagent/result/en/eval_Stage1_en.jsonl"
gui_r1_result = read_jsonl(result_path)

filename = os.path.basename(result_path)  # 得到文件名 'eval_UI_R1_E_zh.jsonl'
if filename.startswith("eval"):
    core_name = filename[len("eval"):]  # 去掉前缀
else:
    core_name = filename
if core_name.endswith(".jsonl"):
    core_name = core_name[:-len(".jsonl")]  # 去掉后缀
core_name = core_name.lstrip('_')

SR = 0
ISR = 0
SCORE = 0

for data in gui_r1_result:
    if data['status'] == "success":
        SR += 1
    if data['inquiry'] == "success":
        ISR += 1
    SCORE += data['reward'] * 4

total = len(gui_r1_result)
SR_percent = SR / total * 100
ISR_percent = ISR / total * 100
SCORE_avg = SCORE / total


print(f"Eval file key: {core_name}")
print(f"ISR: {ISR_percent:.1f}%")
print(f"SR: {SR_percent:.1f}%")
print(f"SCORE: {SCORE_avg:.2f}")


Eval file key: Stage1_en
ISR: 37.9%
SR: 1.1%
SCORE: 0.27


In [18]:
(0.54+0.41)/2

0.475

In [9]:
tool_call_dict = {
  "type": "function",
  "function": {
    "name_for_human": "mobile_use",
    "name": "mobile_use",
    "description": "Use a touchscreen to interact with a mobile device, and take screenshots.\n* This is an interface to a mobile device with touchscreen. You can perform actions like clicking, typing, swiping, etc.\n* Some applications may take time to start or process actions, so you may need to wait and take successive screenshots to see the results of your actions.\n* The screen's resolution is x.\n* Make sure to click any buttons, links, icons, etc with the cursor tip in the center of the element. Don't click boxes on their edges unless asked.",
    "parameters": {
      "properties": {
        "action": {
          "description": "The action to perform. The available actions are:\n* `key`: Perform a key event on the mobile device.\n    - This supports adb's `keyevent` syntax.\n    - Examples: \"volume_up\", \"volume_down\", \"power\", \"camera\", \"clear\".\n* `click`: Click the point on the screen with coordinate (x, y).\n* `long_press`: Press the point on the screen with coordinate (x, y) for specified seconds.\n* `swipe`: Swipe from the starting point with coordinate (x, y) to the end point with coordinates2 (x2, y2).\n* `type`: Input the specified text into the activated input box.\n* `system_button`: Press the system button.\n* `wait`: Wait specified seconds for the change to happen.\n* `terminate`: Terminate the current task and report its completion status.\n* `call_user`: Request user intervention with provided content.",
          "enum": [
            "key",
            "click",
            "long_press",
            "swipe",
            "type",
            "system_button",
            "wait",
            "terminate",
            "call_user"
          ],
          "type": "string"
        },
        "coordinate": {
          "description": "(x, y): The x (pixels from the left edge) and y (pixels from the top edge) coordinates to move the mouse to. Required only by `action=click`, `action=long_press`, and `action=swipe`.",
          "type": "array"
        },
        "coordinate2": {
          "description": "(x, y): The x (pixels from the left edge) and y (pixels from the top edge) coordinates to move the mouse to. Required only by `action=swipe`.",
          "type": "array"
        },
        "text": {
          "description": "Required only by `action=key`, `action=type`.",
          "type": "string"
        },
        "content": {
          "description": "Required only by `action=call_user`.",
          "type": "string"
        },
        "time": {
          "description": "The seconds to wait. Required only by `action=long_press` and `action=wait`.",
          "type": "number"
        },
        "button": {
          "description": "Back means returning to the previous interface, Home means returning to the desktop, Menu means opening the application background menu, and Enter means pressing the enter. Required only by `action=system_button`",
          "enum": [
            "Back",
            "Home",
            "Menu",
            "Enter"
          ],
          "type": "string"
        },
        "status": {
          "description": "The status of the task. Required only by `action=terminate`.",
          "type": "string",
          "enum": [
            "success",
            "failure"
          ]
        }
      },
      "required": [
        "action"
      ],
      "type": "object"
    },
    "args_format": "Format the arguments as a JSON object."
  }
}

system_prompt =  (
    "You are a mobile GUI agent. You are given a task and your action history, with the current screenshot and the previous state preceding the last action. You need to perform the next action to complete the task.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n\n"
    "<tools>\n"
    f"{json.dumps(tool_call_dict, ensure_ascii=False, indent=2)}\n"
    "</tools>\n"
    "For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call>\n\n在标签<think> </think>内输出思考过程。\n在标签<tool_call> </tool_call>内输出最终答案。\n"
)

system_prompt

with open("/home/aiqihang.aqh/Appagent/prompt/train.txt", 'w', encoding='utf-8') as f:
    f.write(system_prompt)